<a href="https://colab.research.google.com/github/cindylimsy/NewDLcatalogue_PNR1z/blob/main/NewDLcatalogue_PNR1z_examplecode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Microseismic workflow on PNR-1z data

#### Cindy Lim Shin Yee
###### Last updated: 23rd Jan 2026
---

The microseismic workflow in this notebook is developed by Cindy Lim Shin Yee for the publication: "A New Deep Learning Enhanced Induced Seismicity
Catalogue at Preston New Road-1z, UK: Improved
Completeness and New Structural Interpretations" on Seismica.

The workflow contains 6 steps:
1. Pre-processing
2. Deep Learning Phase Picking
3. Phase Association
4. Event Location
5. Event Validation
6. Moment Magnitude Estimation

Input:
The seismic dataset provided is one hour of 2000 Hz continuous borehole data from the PNR-1z dataset.

Output:
Validated event catalogue with event origin times, longitude, latitude, depth, Easting and Northing coordinates, moment magnitudes.


# Install and import modules

In [ ]:
!pip install obspy
!pip install pyproj
!pip install seisbench
!pip install pyocto
!pip install multitaper
!pip install NonLinLocPy
!pip install scipy
!pip install xdas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2025.12.0
    Uninstalling dask-2025.12.0:
      Successfully uninstalled dask-2025.12.0


In [ ]:
# import all the modules needed
import string
import time
import argparse as ap
import sys
import os
import numpy as np
import obspy.core as oc
from obspy.core import Stream
from obspy.core import Trace
from obspy.signal.trigger import trigger_onset
from obspy.core import read
import obspy.signal
from obspy.clients.fdsn import Client
import math
import matplotlib as mpl
import pylab as plt
import pandas as pd
from obspy import UTCDateTime
import glob
import h5py
import random
import scipy
import pandas as pd
import datetime as datetime
import pyproj
from scipy.interpolate import InterpolatedUnivariateSpline
import seisbench.models as sbm
from collections import Counter
import pyocto
from pyproj import CRS, Transformer
import re
import seaborn as sns
from copy import deepcopy
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
import datetime
from scipy.interpolate import InterpolatedUnivariateSpline
from scipy.io import loadmat
from obspy import UTCDateTime
import math
from obspy import Trace, Stream, read
import scipy
from scipy.integrate import cumulative_trapezoid
from scipy.fftpack import fft, ifft, fft2, ifft2, fftshift, ifftshift, fftfreq
from scipy import signal
from scipy import ndimage
from math import ceil
from multitaper import MTSpec
from NonLinLocPy import read_nonlinloc
import warnings
import datetime as dt
from autoMwfx import latlon2xxyy, calculate_distance, check_snr, coda_duration, round_seconds, normalized_weighted_average
from pnr import
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Define folder paths
drive = './PNR_run/data/'
savedrive = './'
path_to_velocitymodels = './'

# upload the stations data
stns = pd.read_csv('PNR1z_stations_with_azi_and_inc.csv',index_col=0)

In [ ]:
# Download data and files (takes like 5 minutes)
!git clone https://github.com/cindylimsy/PNR_run.git


Cloning into 'PNR_run'...
remote: Enumerating objects: 2747, done.
remote: Counting objects: 100% (463/463), done.
remote: Compressing objects: 100% (440/440), done.
remote: Total 2747 (delta 161), reused 264 (delta 21), pack-reused 2284 (from 1)
Receiving objects: 100% (2747/2747), 4.93 GiB | 21.76 MiB/s, done.
Resolving deltas: 100% (273/273), done.
Updating files: 100% (1012/1012), done.


In [ ]:
"""## Hyper-parameters"""
print("====================Hyperparameters====================")

# Define data paths
this_folder = "181211"
hour = "09"
filenames1 = int('20'+this_folder+hour)
filenames = drive + this_folder[0:6] + "/" + str(filenames1) + "*.segy"
fhour = glob.glob(filenames)
fhour.sort()

# filter data (set to yes at 50 Hz highpass for this case study)
filter_data = 1
freq_max = 50.0

====================Hyperparameters====================


# Workflow

## Pre-processing

The pre-processing methods followed in this paper (Lim et al., 2026, in submission)

In [ ]:
print("====================DataLoading====================")

# PRE-ALLOCATE instead
ahour = np.zeros((24,2000*16*len(fhour),3))

print('this_folder:', this_folder)
print('filenames1:' , filenames1)

for j in np.arange(0,len(fhour)):
    sthour = readPNR_ENZ(fhour[j], stns, unpack_trace_headers = True)
    if j ==0:
        firststarttime = sthour[0].stats.starttime
    if j==len(fhour)-1:
        lastendtime = sthour[0].stats.endtime
    for i in np.arange(0,len(sthour)):
        ahour[math.floor(i/3),np.arange(j*2000*16,(j+1)*2000*16),(i%3)] = sthour[i].data

# TIME MATRIX TOTAL
t = np.arange(0,len(fhour)*16,0.0005) # from 0 to 3600 seconds, 0.0005 interval
st = sthour
a = ahour
dt = 0.0005

stations = list(set([st[s].stats.station for s in np.arange(0, len(st))]))
stations.sort()

# INPUT FIRST FILE NUMBER NAME
name = fhour[0]

# set starttime
for i in np.arange(0,len(st)):
    st[i].stats.starttime = UTCDateTime(int(name[-23:-19]), int(name[-19:-17]), int(name[-17:-15]), int(name[-15:-13]), int(name[-13:-11]), int(name[-11:-9]))

for s in np.arange(0,len(stations)):
    st2 = st.select(station=stations[s])

chan = st2[0].stats.channel
sr = st2[0].stats.sampling_rate
dt = st2[0].stats.delta
net = st2[0].stats.network
sta = st2[0].stats.station

# Detrending
for i in np.arange(0,(ahour.shape[0])-1):
    a[i,:,0] = obspy.signal.detrend.simple(a[i,:,0])
    a[i,:,1] = obspy.signal.detrend.simple(a[i,:,1])
    a[i,:,2] = obspy.signal.detrend.simple(a[i,:,2])

# filter data matrix
if filter_data:
    import obspy.signal.filter as obs
    for i in np.arange(0,(ahour.shape[0])-1):
        a[i,:,0] = obs.highpass(a[i,:,0],freq=freq_max,df=st[0].stats.sampling_rate)
        a[i,:,1] = obs.highpass(a[i,:,1], freq=freq_max, df=st[0].stats.sampling_rate)
        a[i,:,2] = obs.highpass(a[i,:,2], freq=freq_max, df=st[0].stats.sampling_rate)




====================DataLoading====================
this_folder: 181211
filenames1: 2018121109


In [ ]:
## Make ./mseeds/ and its datafolder (takes ~2 minutes)

# make folder for date + hour
try:
    if (int(str(filenames1)[8:])) < 10:
        try:
            os.mkdir(savedrive + '/mseeds/'+str(this_folder)+'-00'+str(int(str(filenames1)[8:]))+'/')
        except:
            pass
        input_dir = savedrive + '/mseeds/'+str(this_folder)+'-00'+str(int(str(filenames1)[8:]))+'/'
    else:
        try:
            os.mkdir(savedrive + '/mseeds/'+str(this_folder)+'-0'+str(int(str(filenames1)[8:]))+'/')
        except:
            pass
        input_dir = savedrive + '/mseeds/'+str(this_folder)+'-0'+str(int(str(filenames1)[8:]))+'/'
except:
    pass

seq_num = int(str(filenames1)[8:])

for i in np.arange(0, 24):
    stage = i + 1  # stages 1–24

    # zero-padded chunk from filenames1: -001, -010, etc.
    folder_suffix = f"{seq_num:03d}"

    # PR subfolder: PR001, PR010 etc.
    pr_suffix = f"PR{stage:03d}"

    # final path
    dir_path = os.path.join(input_dir, pr_suffix)
    # create all missing parents, ignore if it exists
    os.makedirs(dir_path, exist_ok=True)
    print("Created:", dir_path)

print('Input_dir:',input_dir)

# Each station has their own folder and contains 1 mseed file.

# OUTPUT: 24 MSEED files= 24 stations, 3 components
firststarttime = sthour[0].stats.starttime
for i in np.arange(0,len(sthour)):
    stats = {'network': sthour[i].stats.network, 'station': sthour[i].stats.station, 'starttime': firststarttime, 'endtime': lastendtime,
             'location': '', 'channel': sthour[i].stats.channel, 'npts': a.shape[1], 'sampling_rate': 2000, 'delta': 1/2000,
         'mseed': {'dataquality': 'D'},
         'distance': sthour[i].stats.distance, 'segy': sthour[i].stats.segy}

    if i%3 == 0:
        tr = Trace(a[i//3,:,i%3],header=stats)
    if i%3 == 1:
        tr1 = Trace(a[i//3,:,i%3],header=stats)
    if i%3 == 2:
        tr2 = Trace(a[i//3,:,i%3],header=stats)

        st = Stream([tr,tr1,tr2])
        # write as MSEED file (encoding=0)
        # Stations 1 to 9
        if ((i//3)+1) <= 9:
            print('Station '+ str((i//3)+1))
            st.write(input_dir+"PR00"+str((i//3)+1)+"/PR.PR00"+ str((i//3)+1) + ".." + str(stats['channel']) + "__" + str(firststarttime)[0:13] + str(firststarttime)[14:16] + str(firststarttime)[17:19] + "Z" +"__" + str(lastendtime)[0:13] + str(lastendtime)[14:16] + str(lastendtime)[17:19] + "Z" +  ".mseed", format='MSEED')

        # Stations 10 to 24
        else:
            print('Station '+ str((i//3)+1))
            st.write(input_dir+"PR0"+str((i//3)+1)+"/PR.PR0"+ str((i//3)+1) + ".." + str(stats['channel']) + "__" + str(firststarttime)[0:13] + str(firststarttime)[14:16] + str(firststarttime)[17:19] + "Z" +"__" + str(lastendtime)[0:13] + str(lastendtime)[14:16] + str(lastendtime)[17:19] + "Z" +  ".mseed", format='MSEED')

# Collect all mseed files from PR001–PR024
files = []

for i in range(1, 24+1):
    pr = f"PR{i:03d}"             # PR001 ... PR012
    pr_dir = os.path.join(input_dir, pr)
    files.extend(glob.glob(os.path.join(pr_dir, "*.mseed")))

print("Found files:", len(files))

# Read into a single ObsPy Stream
stream = Stream()
for f in files:
    stream += read(f)
print(stream)

Created: .//mseeds/181211-009/PR001
Created: .//mseeds/181211-009/PR002
Created: .//mseeds/181211-009/PR003
Created: .//mseeds/181211-009/PR004
Created: .//mseeds/181211-009/PR005
Created: .//mseeds/181211-009/PR006
Created: .//mseeds/181211-009/PR007
Created: .//mseeds/181211-009/PR008
Created: .//mseeds/181211-009/PR009
Created: .//mseeds/181211-009/PR010
Created: .//mseeds/181211-009/PR011
Created: .//mseeds/181211-009/PR012
Created: .//mseeds/181211-009/PR013
Created: .//mseeds/181211-009/PR014
Created: .//mseeds/181211-009/PR015
Created: .//mseeds/181211-009/PR016
Created: .//mseeds/181211-009/PR017
Created: .//mseeds/181211-009/PR018
Created: .//mseeds/181211-009/PR019
Created: .//mseeds/181211-009/PR020
Created: .//mseeds/181211-009/PR021
Created: .//mseeds/181211-009/PR022
Created: .//mseeds/181211-009/PR023
Created: .//mseeds/181211-009/PR024
Input_dir: .//mseeds/181211-009/
Station 1
Station 2
Station 3
Station 4
Station 5
Station 6
Station 7
Station 8
Station 9
Station 10
St

## Deep learning phase picking

The PhaseNet model in this notebook is developed by Zhu and Beroza (2018). Original paper: https://doi.org/10.1093/gji/ggy423
More information and tutorials about this model can be found here: https://github.com/AI4EPS/PhaseNet

In [ ]:
# Run PhaseNet (takes about 13 minutes to run on the 1hr of test data)
picker = sbm.PhaseNet.from_pretrained("original")
picker.sampling_rate = 2000

# We used the threshold from Zhu & Beroza 2018 and Lim et al. 2025.
picks = picker.classify(
    stream, batch_size=256, P_threshold=0.3, S_threshold=0.3
).picks

counts = Counter([p.phase for p in picks])
print(
    "P picks:", counts["P"], "\t\tS picks:", counts["S"], "\n"
)  # Output number of P and S picks
print(picks)

# save picks into a dataframe
picks_tosave = picks.to_dataframe()
picks_tosave.to_csv(savedrive+'PhaseNet_Picks_'+str(filenames1)+'.csv')

NameError: name 'sbm' is not defined

## Phase Association
The PyOcto software in this notebook is developed by Münchmeyer (2024). Original paper: https://dx.doi.org/10.26443/seismica.v3i1.1130.
More information and tutorials about this software can be found here: https://pyocto.readthedocs.io/en/latest/

In [ ]:
# upload velocity and density models used in this study
layers = pd.read_csv(path_to_velocitymodels + '/Velocity_models_for_publication/PNR1z_mean_VelModel.csv',index_col=0)
layers['vp'] = layers['VpTop']
layers['vs'] = layers['VsTop']

In [ ]:
# takes 13 seconds
model_path = "velocity_model"
pyocto.VelocityModel1D.create_model(layers, 0.025, 121, 121, model_path)
velocity_model = pyocto.VelocityModel1D(model_path, tolerance=0.05)
velocity_model

In [ ]:
stations = pd.read_csv(path_to_velocitymodels + 'PNR1z_stations_NLL.csv',index_col=0)
stations = stations.drop('station',axis=1)
stations =stations.drop('GTSRCE',axis=1)
stations =stations.drop('label',axis=1)
stations =stations.drop('x_srce',axis=1)
stations =stations.drop('y_srce',axis=1)
stations =stations.drop('z_srce',axis=1)
stations =stations.drop('elev',axis=1)
# change the station elevations to positive and in km
stations['elevation'] = -stations['elevation']/1000

In [ ]:
# Set latlon from NonLinLoc setting: TRANS SIMPLE 53.77577 -2.98778 0
latitude = 53.77577
longitude = -2.98778

associator = pyocto.OctoAssociator.from_area(
    lat=(latitude, latitude + (3/111)),
    lon=(longitude, longitude + (3/111)),
    zlim=(0, 3),
    time_before=0.3,
    velocity_model=velocity_model,
    min_node_size = 0.3,
    min_node_size_location = 0.1,
    pick_match_tolerance= 0.05,
    min_interevent_time = 0,
    n_p_and_s_picks=4,
    min_pick_fraction = 0
)

associator.transform_stations(stations)
stations

In [ ]:
# will take
# upload the picks
picks = picks_tosave

# ensure all time is UTCDateTime
picks['time'] = picks['time'].apply(UTCDateTime)
picks['UTCDateTime'] = picks['time']

# Convert the UTCDateTime column to timestamps
picks['time'] = picks['UTCDateTime'].apply(lambda x: x.timestamp)
# formatting 'station' column so it is PRXX
picks['station'] = 'PR' + picks_tosave["station"].str.slice(3, -1).astype(int).astype(str).str.zfill(2)
picks['phase'] = picks['phase'].str.upper()

picks.drop('probability',axis=1,inplace=True)
picks.drop('UTCDateTime',axis=1,inplace=True)
picks.drop('index',axis=1,inplace=True)

# Define the desired column order
desired_order = ['station', 'phase', 'time']

# Rearrange the DataFrame columns
picks = picks[desired_order]

print('Length of picks:', len(picks))

startnow = datetime.datetime.now()
events, assignments = associator.associate(picks, stations)
endnow = datetime.datetime.now()
time_taken = endnow - startnow
print('Time taken:', time_taken)

print('Length of Events:',len(events))

print('Length of Events:',len(assignments))

assignments["time"] = assignments["time"].apply(datetime.datetime.fromtimestamp, tz=datetime.timezone.utc)
pyocto.OctoAssociator.to_nonlinloc(assignments,'./PhaseNet_pyocto_NLL_file_'+str(filenames1)+'.hpf')
print(str(filenames1) + ' saved and done.')

## Event Validation

The Linear Moveout Event Filter (LinMEF) event validation code is developed by Lim et al. (2025). Original paper: https://doi.org/10.1093/gji/ggae386

In [ ]:
assignments

In [ ]:
# define the index
df_tocheck = assignments
numevent = np.arange(0, assignments['event_idx'].max())
# numevent = np.arange(0, 1)

widths = []
names = []
gradients = []
labels = []
# if no picks in the file, add to a list
nopicks = []

for event in numevent:
    print('Progress: ' + str(event) + ' of ' + str(assignments['event_idx'].max()))

    # first get picks
    hpf_file = assignments[assignments['event_idx']==event]
    hpf_file['time'] = hpf_file['time'].apply(UTCDateTime)
    # first get the all the p picks past the starttime

    if len(hpf_file) > 0:
        pass
    else:
        nopicks.append(event)
        print('No picks at event ', event)
        continue

    # get the minimum time
    t0 = np.min(hpf_file['time'])
    hpf_file.sort_values(by='station')
    timediff = []

    # get the timediff between the earliest time after origin time and each P pick
    for i in np.arange(0, len(hpf_file)):
        timediff.append(t0 - hpf_file['time'].iloc[i])

    hpf_file['stz'] = [''] * len(hpf_file)

    # get station depth (i.e. the y coords) for hpf_file
    droplist = []
    hpf_file = hpf_file.reset_index()
    for i in np.arange(0, len(hpf_file)):
        stz = stns['stz'][int(hpf_file['station'].iloc[i][2:]) == stns['name']]
        hpf_file['stz'].iloc[i] = stz

    # delete any abs(picks-median)> 0.1, <0.1
        if abs(timediff[i] - np.median(np.array(timediff))) <= 0.1:
            pass
        else:
            # drop it
            droplist.append(i)

    hpf_file.drop(droplist, axis=0, inplace=True)

    if len(hpf_file) > 0:
        pass
    else:
        nopicks.append(event)
        print('No picks at event', event)
        continue

    timediff = []

    for i in np.arange(0, len(hpf_file)):
        timediff.append(t0 - hpf_file['time'].iloc[i])

    print(str(event) + '. Generating line of best fit...')
    timediff = [float(i) for i in timediff]
    stz_list = hpf_file['stz'].values[0:len(hpf_file)]
    stz = [float(i) for i in stz_list]
    # L1 norm calculation
    x = np.array(stz)
    b = np.array(timediff)
    A = np.vstack([x, np.ones(len(x))]).T
    m, c = np.linalg.lstsq(A, b, rcond=None)[0]
    aL2 = m

    loss = lambda ps: np.linalg.norm(ps[0] * x + ps[1] - b, ord=1)

    best_loss = 1000000000
    for i in np.arange(0, 10):
        params = minimize(loss, np.array([random.uniform(0, 5), random.uniform(0, -5)]))
        current_loss = loss(params.x)
        if current_loss < best_loss:
            best_loss = current_loss
            best_params = params.x

    # params[0] is the gradient and
    # params[1] will be the y intercept, after minimizing l1 loss
    params = best_params
    aL1 = params[0]
    cL1 = params[1]

    # save the gradients
    gradients.append(aL1)

    # add line of best fit to plot
    newtimediffL2 = aL2 * np.array(stz) + c
    newtimediffL1 = aL1 * np.array(stz) + cL1


    t_residuals = np.array(newtimediffL1) - np.array(timediff)

    # get the percentiles of the residuals
    p95 = np.percentile(t_residuals, 95)
    p5 = np.percentile(t_residuals, 5)
    p25 = np.percentile(t_residuals, 25)
    p75 = np.percentile(t_residuals, 75)

    width = p95 - p5
    widths.append(width)
    names.append(event)

    # label conditions here
    if (aL1 >= 0.00021) and (aL1 <= 0.00031):
        labels.append('TP')
    else:
        labels.append('nonTP')


    print(str(event+1) + '/' + str(assignments['event_idx'].max()) + ' done.')

# save list of no picked events
nopicks_df = pd.DataFrame(columns=['Time'])
nopicks_df['Time'] = nopicks
nopicks_df.to_csv(savedrive + 'nopicks_forevents.csv')

# make a dataframe of the events plus their widths
widths_df_final2 = pd.DataFrame(columns=['Time', 'Widths','Gradients','label'])
widths_df_final2['Time'] = names
widths_df_final2['Widths'] = widths
widths_df_final2['Gradients'] = gradients
widths_df_final2['label'] = labels
widths_df_final2.to_csv(savedrive + 'LinMEF_labelled_L1_gradients.csv')

# plot histplot of the total df
plt.figure(figsize=(10, 8))
sns.distplot(a=widths_df_final2['Widths'], hist=True)
plt.axvline(x=np.nanmedian(widths_df_final2['Widths']),
            color='black',linestyle='dashed')
plt.tight_layout()
plt.savefig(savedrive+ 'Total_TWidths.png')
plt.show()

# plot of the labels
plt.figure(figsize=(10, 8))
sns.distplot(a=widths_df_final2[widths_df_final2['label'] =='TP']['Widths'], hist=True)
sns.distplot(a=widths_df_final2[widths_df_final2['label'] !='TP']['Widths'], hist=True)
plt.axvline(x=np.nanmedian(widths_df_final2[widths_df_final2['label'] =='TP']['Widths']),
            color='blue',linestyle='dashed')
plt.axvline(x=np.nanmedian(widths_df_final2[widths_df_final2['label'] !='TP']['Widths']),
            color='orange',linestyle='dashed')
plt.legend(['TP','non TP', 'TP median', 'non TP median'])
plt.tight_layout()
plt.savefig(savedrive+ 'TWidths_labelled.png')
plt.show()
# plt.close()

# plot the ECDF of total df
plt.figure(figsize=(10, 8))
sns.ecdfplot(data=widths_df_final2['Widths'])
plt.axhline(y=0.5, linestyle='dashed')
plt.tight_layout()
plt.savefig(savedrive+ 'ECDF_Total_TWidths.png')
plt.show()


# plot the ECDF of labels
sns.ecdfplot(data=widths_df_final2[widths_df_final2['label'] =='TP']['Widths'])
sns.ecdfplot(data=widths_df_final2[widths_df_final2['label'] !='TP']['Widths'])
plt.legend(['TP', 'non TP'])
plt.axhline(y=0.5, linestyle='dashed')
plt.tight_layout()
plt.savefig(savedrive+ 'ECDF_TWidths_labelled.png')
plt.close()
# plt.show()

# plot total gradients
plt.figure(figsize=(14, 8))
sns.distplot(widths_df_final2['Gradients'],label='distribution')
plt.axvline(x=np.nanmedian(widths_df_final2['Gradients']),
            color='blue',label='median')
plt.xlabel('Linear regression slope')
plt.tight_layout()
plt.savefig(savedrive+'Total_gradients.png')
plt.show()

# from the gradients, plot the ones that say TP
plt.figure(figsize=(14, 8))
sns.distplot(widths_df_final2['Gradients'][widths_df_final2['label']=='TP'],label='TP')
sns.distplot(widths_df_final2['Gradients'][widths_df_final2['label']!='TP'],label='non-TP')
plt.axvline(x=np.nanmedian(widths_df_final2[widths_df_final2['label'] =='TP']['Gradients']),
            color='blue',label='TP median')
plt.axvline(x=np.nanmedian(widths_df_final2[widths_df_final2['label'] !='TP']['Gradients']),
            color='orange', label='non-TP median')
plt.xlabel('Linear regression slope')
plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(savedrive+'Total_gradients_labelled.png')
plt.show()

# ecdf plot of the total gradients
plt.figure(figsize=(12, 8))
sns.ecdfplot(widths_df_final2['Gradients'],label='TP')
plt.axhline(y= 0.5, color='black', linestyle='--')
plt.xlabel('Linear regression slope')
plt.legend()
plt.tight_layout()
plt.savefig(savedrive+'ECDF_Total_gradients.png')
plt.show()

# ecdf plot of the labelled gradients
plt.figure(figsize=(12, 8))
sns.ecdfplot(widths_df_final2['Gradients'][widths_df_final2['label']=='TP'],label='TP')
sns.ecdfplot(widths_df_final2['Gradients'][widths_df_final2['label']!='TP'],label='non-TP')
plt.axhline(y= 0.5, color='black', linestyle='--')
plt.xlabel('Linear regression slope')
plt.legend()
plt.tight_layout()
plt.savefig(savedrive+'ECDF_Total_gradients_labelled.png')
plt.show()

In [ ]:
assignments_filtered = assignments[assignments["event_idx"].isin(widths_df_final2["Time"])]

## Event Location

The NonLinLoc software is developed by Lomax et al. (2000). Original paper: https://doi.org/10.1007/978-94-015-9536-0_5

In [ ]:
%%bash
apt-get update -y
apt-get install -y \
  git \
  cmake \
  build-essential \
  gfortran

cd /content
git clone https://github.com/alomax/NonLinLoc.git

cd /content/NonLinLoc/src

# Remove any existing bin (or broken symlink)
rm -rf bin

# Create bin inside src (recommended for Colab)
mkdir bin

cd /content/NonLinLoc/src
rm -f CMakeCache.txt

cd /content/NonLinLoc/src
cmake .
cd /content/NonLinLoc/src
make -j2

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,885 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:14 http

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'NonLinLoc'...
/content/NonLinLoc/src/GridLib.c: In function ‘ReadGrid3dHdr’:
/content/NonLinLoc/src/GridLib.c:2210:9: warning: ignoring return value of ‘fscanf’ declared with attribute ‘warn_unused_result’ [-Wunused-result]
 2210 |         fscanf(fpio, "%s %lf %lf %lf\n",
      |         ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 2211 |             psrce->label, &(psrce->x), &(psrce->y), &(psrce->z));
      |             ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/content/NonLinLoc/src/GridLib.c: In function ‘OpenGrid3dFile’:
/content/NonLinLoc/src/GridLib.c:2368:9: warning: ignoring return value of ‘fscanf’ declared with attribute ‘warn_unused_result’ [-Wunused-result]
 2368 |         fscanf(*fp_hdr, "%s %lf %lf %lf\n",
      |         ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 2369 |      

In [ ]:
!mkdir ./NLLoc
!mkdir ./NLLoc/picks
!mkdir ./NLLoc/model
!mkdir ./NLLoc/time
!mkdir ./NLLoc/loc

In [ ]:
# blanketPScindy_nll.in (I just uploaded the Vel2grid to ./NLLoc/model/)
!./NonLinLoc/src/bin/Vel2Grid ./nlloc_control_vel2grid.in

Vel2Grid (NonLinLoc v7.1.05 18Apr2025) 
CONTROL:  MessageFlag: 3  RandomNumSeed: 54321
TRANSFORM  SIMPLE LatOrig 53.775770  LongOrig -2.987780  RotCW 0.000000
Vel2Grid files:  Output: ./NLLoc/model/3km0.025.*
Vel2Grid wave type:  P
Vel2Grid wave type:  S
GRID: {x, y, z}
  Num: {121, 121, 121}
  Orig: {0, 0, 0}
  LenSide: {0.025, 0.025, 0.025}
  Type: SLOW_LEN


In [ ]:
# Pg2tblanketPScindy_nll.in
! ./NonLinLoc/src/bin/Grid2Time ./nlloc_control_grid2time_P.in
# Sg2tblanketPScindy_nll.in
! ./NonLinLoc/src/bin/Grid2Time ./nlloc_control_grid2time_S.in

Streaming output truncated to the last 5000 lines.
update side z=108: ff fb bb bf z=108 <R#1>updated.
update side z=107: ff fb bb bf z=107 <R#1>updated.
update side z=106: ff fb bb bf z=106 <R#1>updated.
update side z=105: ff fb bb bf z=105 <R#1>updated.
update side z=104: ff fb bb bf z=104 <R#1>updated.
update side z=103: ff fb bb bf z=103 <R#1>updated.
update side z=102: ff fb bb bf z=102 <R#1>updated.
update side z=101: ff fb bb bf z=101 <R#1>updated.
update side z=100: ff fb bb bf z=100 <R#1>updated.
update side z=99: ff fb bb bf z=99 <R#1>updated.
update side z=98: ff fb bb bf z=98 <R#1>updated.
update side z=97: ff fb bb bf z=97 <R#1>updated.
update side z=96: ff fb bb bf z=96 <R#1>updated.
update side z=95: ff fb bb bf z=95 <R#1>updated.
update side z=94: ff fb bb bf z=94 <R#1>updated.
update side z=93: ff fb bb bf z=93 <R#1>updated.
update side z=92: ff fb bb bf z=92 <R#1>updated.
update side z=91: ff fb bb bf z=91 <R#1>updated.
update side z=90: ff fb bb bf z=90 <R#1>updated.


In [ ]:
import datetime

In [ ]:
# NLLOC from code before (takes 16 minutes)
start = datetime.datetime.now()
!./NonLinLoc/src/bin/NLLoc ./nlloc_control_nlloc.in
end = datetime.datetime.now()
timing = end - start
print(timing)


Streaming output truncated to the last 5000 lines.

Delayed, Sorted, Centered Observations:
    0  PR24         P      09:59:34.0180 -  0.0000s ->  -0.0907 (35974.0180)
    1  PR23         P      09:59:34.0205 -  0.0000s ->  -0.0882 (35974.0205)
    2  PR21         P      09:59:34.0325 -  0.0000s ->  -0.0762 (35974.0325)
    3  PR20         P      09:59:34.0385 -  0.0000s ->  -0.0702 (35974.0385)
    4  PR19         P      09:59:34.0440 -  0.0000s ->  -0.0647 (35974.0440)
    5  PR18         P      09:59:34.0505 -  0.0000s ->  -0.0582 (35974.0505)
    6  PR17         P      09:59:34.0580 -  0.0000s ->  -0.0507 (35974.0580)
    7  PR24         S      09:59:34.0615 -  0.0000s ->  -0.0472 (35974.0615)
    8  PR23         S      09:59:34.0700 -  0.0000s ->  -0.0387 (35974.0700)
    9  PR16         P      09:59:34.0705 -  0.0000s ->  -0.0382 (35974.0705)
   10  PR22         S      09:59:34.0780 -  0.0000s ->  -0.0307 (35974.0780)
   11  PR14         P      09:59:34.0800 -  0.0000s ->  -0.02

In [ ]:
import glob
glob.glob('./NLLoc/loc/*sum*')

['./NLLoc/loc/ALLPSpyocto.sum.grid0.loc.arc',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hdr',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.stations',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hyp',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.stat_totcorr',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hypo_71',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.stat',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hypo_inv',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.csv',
 './NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hypo_ell']

In [ ]:
from google.colab import files
files.download('./NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hyp')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
##############################
# NLLOC output --> catalogue
##############################

import glob
# path to
origins = []
lats = []
lons = []
depths = []

with open('./NLLoc/loc/ALLPSpyocto.sum.grid0.loc.hyp','r') as fi:
  for ln in fi:
    if ln.startswith('GEOGRAPHIC'):
      origins.append(ln[15:42])
      lats.append(float(ln[47:57]))
      lons.append(float(ln[62:72]))
      depths.append(float(ln[78:]))

# loop through and convert origin times to UTCDateTime
for i in range(len(origins)):
  origins[i] = UTCDateTime(origins[i][:4] + origins[i][5:7] + origins[i][8:10] + 'T' + origins[i][12:14] + origins[i][15:17] + "%.4f" % float(origins[i][18:27].zfill(7)))

# create a dataframe of event locations
dfs = pd.DataFrame(list(zip(origins,lats,lons,depths)), columns = ['time', 'lat', 'lon', 'depth'])

# dfs = glob.glob(root + 'NLLoc/df_events*')

# Define the wgs84 and osgb36 projection
wgs84 = pyproj.CRS("EPSG:4326")
osgb = pyproj.CRS("EPSG:27700")

lat = dfs['lat']
lon = dfs['lon']
xx, yy = pyproj.transform(wgs84,osgb,lat,lon)
dfs['xx'] = xx
dfs['yy'] = yy
dfs.to_csv('./df_catalogue.csv')



## Magnitude Estimation

This is the method to estimate moment magnitudes developed in this paper (Lim et al., 2026, in submission)

In [ ]:
dfs = pd.read_csv('./df_catalogue.csv',index_col=0)

In [ ]:
dfs

,time,lat,lon,depth,xx,yy
0,2018-12-11T09:01:06.005000Z,53.788417,-2.961702,1.934659,336738.219856,432884.674390
1,2018-12-11T09:05:08.540000Z,53.789720,-2.969356,2.164773,336235.949170,433036.498301
2,2018-12-11T09:08:56.391100Z,53.787880,-2.967800,2.369318,336335.665285,432830.390314
3,2018-12-11T09:15:05.225700Z,53.789643,-2.965853,2.275568,336466.605228,433024.785995
4,2018-12-11T09:15:08.487600Z,53.787957,-2.966114,2.360795,336446.858005,432837.443923
...,...,...,...,...,...,...
1631,2018-12-11T09:59:55.188200Z,53.787727,-2.965465,2.352273,336489.266975,432811.273543
1632,2018-12-11T09:59:56.813600Z,53.788877,-2.966373,2.360795,336431.188138,432940.030840
1633,2018-12-11T09:59:57.541800Z,53.788877,-2.966113,2.343750,336448.316911,432939.797688
1634,2018-12-11T09:59:59.107800Z,53.788110,-2.965595,2.360795,336481.281961,432854.000724


In [ ]:
#=====================================================#
# User input (folder paths, catalogue)
#=====================================================#
# main code
BP_path = './'
df_touse = dfs

# upload Q_indiv_stn.csv
Q_indiv_stn_df = pd.read_csv(BP_path + 'Q_indiv_stn_300randomevents.csv',index_col=0)
# Defining different Q's
Qstn = Q_indiv_stn_df['Q_stn'].values # list of Q's per station - get this from Q_indiv_stn_df
data_dir =  './'
stns = pd.read_csv(BP_path + '/PNR-1z_Stations_orient.dat', delim_whitespace=True)
#=====================================================#
# uploading date range for hyp file matching
#=====================================================#
# Define the date range
start_date = "2018-10-15"
end_date = "2018-12-18"

# Generate the date range
date_range = pd.date_range(start=start_date, end=end_date)

# Map dates to day numbers
day_numbers = range(1, len(date_range) + 1)

# Create a DataFrame or dictionary for association
date_to_day = pd.DataFrame({"Date": date_range, "DayNumber": day_numbers})
warnings.filterwarnings("ignore")


In [ ]:
# ===================================================
# 0. define plotting switches, dataframe to use
# pre-allocate empty lists to store the results for each station
# ===================================================
# import velocity and density model
mean_velmodel = pd.read_csv(BP_path + 'PNR1z_mean_VelModel.csv',index_col=0)
rho_model = pd.read_csv(BP_path + '121018_PNR-1z_PureGun_01_Calibration.bormod',skiprows=10,delimiter='\t',names=['Depth','Vp','Vs','Rho','Dip','Thomsen_Epsilon','Thomsen_Delta','Symmetry_Axis_Theta','Thomsen_Gamma','nan'])

depths = mean_velmodel["depth"].values*1000
depths_rho = rho_model['Depth'].values # this model's depth is in 1000s already
vs_profile = mean_velmodel["VsTop"].values*1000
rho_profile = rho_model['Rho'].values*1000

# make sure the catalogue is in the correct format
df_touse['datetime'] = df_touse['time'].apply(UTCDateTime)
df_touse['strdatetime'] = df_touse['time'].apply(str)
df_touse['T'] = df_touse['datetime'].apply(round_seconds)
df_touse['T'] = df_touse['T'].apply(str)

# defining columns to fill (Omega0, M0, logM0 and mean_Mw)
df_touse['mean_Mw'] = np.nan * len(df_touse)
df_touse['M0'] = np.nan * len(df_touse)
df_touse['logM0'] = np.nan * len(df_touse)
df_touse['Omega0'] = np.nan * len(df_touse)

# reset index for df_touse
df_touse = df_touse.reset_index(drop=True)
station_nos = np.arange(0,24)
# data type mapping from hpf or hyp file
dtype_mapping = {'station': 'str', 'instrument': 'str',
                 'component': 'str', 'p_onset': 'str', 'phase': 'str',
                 'first_motion': 'str', 'date': 'int', 'hhmm': 'int',
                 'ss': 'float', 'err': 'str', 'errmag': 'float',
                 'coda': 'float', 'amp': 'float', 'period': 'float'}
window_before_after = [0.03, 0.2]

# Apply conversion factor, filter and integrate to velocity or displacement
sensitivity = 115564 # V/ m/s
convert_factor = 1/sensitivity
geophone_natural_freq = 15 # 15 Hz for natural freq
low_freq_plateau_max_range = 100.0 # Hz
bp_filter = [geophone_natural_freq, 850]
savelastdf = 1
filter_data = 0
plot_switch = 0
verbose = 0
plot_spectra = 0

# Radiation pattern for a S-wave
R = 0.63
# Free air correction (set to 1 for borehole waveforms)
F = 1

______

## Try another magnitude estimation with my code

In [ ]:
import pnr
import autoMw_workflow
from pnr import readPNR, readPNR_ENZ
from autoMw_workflow4 import autoMw, AutoMwPaths, AutoMwConfig

<module 'autoMw_workflow4' from '/content/autoMw_workflow4.py'>

In [ ]:
# 2) Your input catalogue dataframe (must have at least 'time')
df_catalogue = dfs.copy()   # your existing dataframe

# 3) Configure paths (edit these to your folder layout)
paths = AutoMwPaths(
    base_path="./",                                  # where metadata csv/dat live + where outputs will be written
    segy_glob="./*.segy",                            # where your SEGY files are
    hyp_glob="./*{timestr}*.hyp",                    # where your NLL hyp files are (same folder in your script)
    q_indiv_csv="Q_indiv_stn_300randomevents.csv",
    stations_file="./PNR1z_stations_with_azi_and_inc.csv",
    mean_velmodel_csv="PNR1z_mean_VelModel.csv",
    rho_model_file="121018_PNR-1z_PureGun_01_Calibration.bormod",
)

# 4) Configure processing (edit anything you previously changed by hand)
cfg = AutoMwConfig(
    start_date="2018-12-11",
    end_date="2018-12-11",
    window_before_after=(0.03, 0.2),
    filter_data=0,  # filter switch
    savelastdf=1, # save last dataframe switch
    save_every_n=10,  # save dataframe every nth event
    verbose=0, # print statements out
    R=0.63,
    F=1.0,
)


In [ ]:
# 5) Run
mw = autoMw(df_catalogue=df_catalogue, paths=paths, cfg=cfg)

# quick test on first few events (10 events takes ~2 minute)
df_out_test = mw.run(max_events=10)

# full run (warning: this will take a while...)
# df_out = mw.run()


EVENT 1
2018-12-11T09:01:06.005000Z
No event file found for Event 1. Skipping...
EVENT 2
2018-12-11T09:05:08.540000Z
EVENT 3
2018-12-11T09:08:56.391100Z
EVENT 4
2018-12-11T09:15:05.225700Z
EVENT 5
2018-12-11T09:15:08.487600Z
EVENT 6
2018-12-11T09:15:10.527300Z
EVENT 7
2018-12-11T09:15:11.733200Z
No event file found for Event 7. Skipping...
EVENT 8
2018-12-11T09:15:42.167600Z
EVENT 9
2018-12-11T09:15:54.599800Z
EVENT 10
2018-12-11T09:15:58.907700Z
